In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np

from src.utils import (
    get_args,
    set_seed,
    get_datesets_and_loaders,
    get_trained_VAE,
    get_trained_VAE_with_domain_classifier,
    get_trained_classifier,
    get_trained_classifier_Base,
    test_model,
    prepare_report,
    run_all_senario
)

/home/asad/workspace/DomainProject/changeDomain/notebooks/effective-gzsda/gzsda/src/utils.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy


In [3]:
DOMAIN_SET =['Art','Clipart','Product','RealWorld']
DATA_DIR = './data/OfficeHome/'
DATASET_DETAILS = {
    "prefix": 'OfficeHome-',
    "suffix": '-resnet50-noft.mat',
    "resnet_feature": 'resnet50_features',
    "split_file_name": 'instanceSplit_officehome_unseen30.mat'
}
NUM_LABELS=65

In [4]:
import json
from pathlib import Path

RESULT_OBJ_PATH = "./result/json/officeHome.json"
RESULT_CSV_PATH = "./result/csv/officeHome.csv"
path = Path(RESULT_OBJ_PATH)

if path.exists():
    with path.open("r", encoding="utf-8") as f:
        result = json.load(f)
else:
    result = {}

result.keys()

dict_keys(['base', 'CCVAE', 'our0', 'our_GRE'])

In [5]:
base = "base"
CCVAE = "CCVAE"
our0 = "our0"
our_GRE = "our_GRE"

# clear last result
# result.pop(base, None)
# result.pop(CCVAE, None)
# result.pop(our0, None)
# result.pop(our_GRE, None)

## Base

In [6]:
def main_base(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    classifier = get_trained_classifier_Base(
        data_loaders=data_loaders,
        NUM_LABELS=NUM_LABELS,
        device=device)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [7]:
if base not in result:
    result[base] = run_all_senario(main_base, DOMAIN_SET)

# GZSDA

In [8]:
def main_gzsda(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [9]:
if CCVAE not in result:
    result[CCVAE] = run_all_senario(main_gzsda, DOMAIN_SET)

## m0

In [10]:
def main_m0(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [11]:
if our0 not in result:
    result[our0] = run_all_senario(main_m0, DOMAIN_SET)

## m1: seperate after encoder

In [12]:
def main_m1(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE_with_domain_classifier(
        data_loaders=data_loaders,
        args=args,
        device=device)
        
    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [13]:
if our_GRE not in result:
    result[our_GRE] = run_all_senario(main_m1, DOMAIN_SET)
    

## Merge results

In [14]:
with open(RESULT_OBJ_PATH, "w") as f:
    json.dump(result, f, indent=2)

In [15]:
# ignore our0
result.pop(our0, None)

{'Art -> Clipart': 'Seen:     64.03 ± 0.32\nUnseen:   45.49 ± 0.67\nH-mean:   53.18 ± 0.45',
 'Art -> Product': 'Seen:     85.36 ± 0.38\nUnseen:   68.37 ± 1.17\nH-mean:   75.89 ± 0.63',
 'Art -> RealWorld': 'Seen:     81.53 ± 0.86\nUnseen:   73.63 ± 0.98\nH-mean:   77.34 ± 0.37',
 'Clipart -> Art': 'Seen:     66.82 ± 1.13\nUnseen:   53.21 ± 0.69\nH-mean:   59.20 ± 0.33',
 'Clipart -> Product': 'Seen:     83.19 ± 0.75\nUnseen:   66.22 ± 2.15\nH-mean:   73.63 ± 1.11',
 'Clipart -> RealWorld': 'Seen:     80.95 ± 0.94\nUnseen:   68.29 ± 1.80\nH-mean:   73.98 ± 0.68',
 'Product -> Art': 'Seen:     65.51 ± 0.59\nUnseen:   53.97 ± 0.46\nH-mean:   59.17 ± 0.33',
 'Product -> Clipart': 'Seen:     63.75 ± 0.24\nUnseen:   46.64 ± 1.29\nH-mean:   53.83 ± 0.90',
 'Product -> RealWorld': 'Seen:     81.00 ± 0.79\nUnseen:   74.92 ± 1.40\nH-mean:   77.78 ± 0.47',
 'RealWorld -> Art': 'Seen:     68.75 ± 0.96\nUnseen:   65.38 ± 1.13\nH-mean:   66.96 ± 0.33',
 'RealWorld -> Clipart': 'Seen:     64.42 ± 0.

In [16]:
import pandas as pd
import re

rows = [(k, m, result[m][k]) for m in result for k in result[m]]
df = pd.DataFrame(rows, columns=['domain', 'method', 'values'])

def extract_metrics(text):
    matches = dict(re.findall(r'(\w+):\s+([\d.]+\s*±\s*[\d.]+)', text))
    return pd.Series(matches)

df[['seen', 'unseen', 'H-mean']] = df['values'].apply(extract_metrics)
df = df[['domain', 'method', 'seen', 'unseen', 'H-mean']]
df['method'] = pd.Categorical(df['method'], categories=[base, CCVAE, our0, our_GRE], ordered=True)
df = df.sort_values(['domain', 'method']).reset_index(drop=True)

df

,domain,method,seen,unseen,H-mean
0,Art -> Clipart,base,71.69 ± 0.55,28.16 ± 0.58,40.41 ± 0.52
1,Art -> Clipart,CCVAE,66.56 ± 0.56,41.56 ± 0.79,51.13 ± 0.48
2,Art -> Clipart,our_GRE,63.98 ± 0.60,45.21 ± 0.76,52.96 ± 0.50
3,Art -> Product,base,89.66 ± 0.14,53.13 ± 1.24,66.68 ± 0.96
4,Art -> Product,CCVAE,87.17 ± 0.38,65.26 ± 0.98,74.62 ± 0.60
5,Art -> Product,our_GRE,85.41 ± 0.48,68.76 ± 1.19,76.15 ± 0.67
6,Art -> RealWorld,base,85.95 ± 0.93,62.81 ± 0.59,72.56 ± 0.37
7,Art -> RealWorld,CCVAE,83.59 ± 0.90,69.13 ± 0.86,75.65 ± 0.39
8,Art -> RealWorld,our_GRE,81.35 ± 1.04,73.35 ± 0.74,77.10 ± 0.27
9,Clipart -> Art,base,72.71 ± 0.46,33.25 ± 1.35,45.56 ± 1.24


In [17]:
df.to_csv(RESULT_CSV_PATH)